# Reproduce the reported IWSLT'14 results (local, CPU, no training)

Rebuilds the final Week 9 study report from the evidence committed in
`experiments/2026-11-22_iwslt14-week9-benchmark/` and checks it against the published copy:

1. downloads IWSLT'14 (pinned URL and SHA-256) and rebuilds the splits and the 10k tokenizer;
2. unpacks the archived runs (configs, logs, evaluations, test hypotheses) into a scratch folder;
3. regenerates `STUDY.md` and `STUDY.json` with `scripts/analyze_study.py` and checks that they are
   **byte-identical** to the committed reports;
4. re-scores all 36 test hypothesis files with sacreBLEU and checks each against its logged BLEU.

**Needs:** the package installed (`pip install -e .`, see README → Development setup), and internet
once for the ~20 MB archive. **No GPU and no checkpoints.**

**Writes only to:** `data/` and `runs/reproduce_week9/`, both gitignored.

**Time:** a few minutes on a laptop CPU, mostly data preparation and bootstrap statistics.

Retraining the benchmark is a different workflow; see `notebooks/README.md`.

In [ ]:
# 0. Parameters
ARCHIVE = "experiments/2026-11-22_iwslt14-week9-benchmark"
STUDY_CONFIG = "configs/study/week9_benchmark.yaml"
DATA_CONFIG = "configs/data/iwslt14.yaml"
TOKENIZER = "data/tokenizers/iwslt14_bpe10000.model"
# Content fingerprint recorded in every benchmark run's metadata.json (ADR-005, ADR-023 §3)
TOKENIZER_FINGERPRINT = "b0c6c22a45b05ef17842a299f3e41bc7ccb971d3bfd90dab8a3858872b261989"
WORK = "runs/reproduce_week9"

In [ ]:
# 1. Locate the repository root and define a command helper
import filecmp
import glob
import gzip
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

T0 = time.time()
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(ROOT)
print("repository:", ROOT)


def run(*args):
    cmd = [sys.executable, *args]
    print("$ python", " ".join(args))
    subprocess.run(cmd, check=True)

In [ ]:
# 2. Data and tokenizer from public sources (checksums and split counts are verified)
run("scripts/prepare_data.py", "--config", DATA_CONFIG)
if not os.path.exists(TOKENIZER):
    run("scripts/train_tokenizer.py", "--config", DATA_CONFIG)
from aiayn.data.tokenizer import SentencePieceTokenizer

fingerprint = SentencePieceTokenizer(TOKENIZER).fingerprint
assert fingerprint == TOKENIZER_FINGERPRINT, (
    f"tokenizer differs from the benchmark's: {fingerprint}"
)
for meta in glob.glob(f"{ARCHIVE}/runs/iwslt14_*/metadata.json"):
    with open(meta, encoding="utf-8") as fh:
        assert json.load(fh)["tokenizer_fingerprint"] == fingerprint, meta
print("tokenizer identical (fingerprint) to the one recorded by every archived run")

In [ ]:
# 3. Unpack the archived runs: decompress hypotheses, add the tokenizer each run expects
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(f"{ARCHIVE}/runs", WORK)
for packed in glob.glob(f"{WORK}/*/eval/*.hyp.gz"):
    with gzip.open(packed, "rb") as src, open(packed[: -len(".gz")], "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.remove(packed)
for run_dir in glob.glob(f"{WORK}/*/"):
    os.makedirs(f"{run_dir}tokenizer", exist_ok=True)
    shutil.copy(TOKENIZER, f"{run_dir}tokenizer/")
print(len(glob.glob(f"{WORK}/*/")), "runs unpacked into", WORK)

In [ ]:
# 4. Regenerate the study report and compare it with the committed one
run(
    "scripts/analyze_study.py", "--study", STUDY_CONFIG, "--work", WORK, "--out", f"{WORK}/STUDY.md"
)
for name in ("STUDY.md", "STUDY.json"):
    same = filecmp.cmp(f"{WORK}/{name}", f"{ARCHIVE}/study/{name}", shallow=False)
    print(f"{name}: {'IDENTICAL' if same else 'DIFFERENT'} to {ARCHIVE}/study/{name}")
    assert same, f"{name} differs: compare the two files"

In [ ]:
# 5. Re-score every archived test hypothesis file against its logged BLEU
import sacrebleu

with open("data/processed/iwslt14/test.en", encoding="utf-8") as fh:
    references = fh.read().splitlines()
rows = []
for hyp_path in sorted(glob.glob(f"{WORK}/*/eval/test.*.hyp")):
    with open(hyp_path, encoding="utf-8") as fh:
        hypotheses = fh.read().splitlines()
    with open(hyp_path[: -len(".hyp")] + ".json", encoding="utf-8") as fh:
        logged = json.load(fh)["bleu"]
    score = round(sacrebleu.corpus_bleu(hypotheses, [references]).score, 2)
    rows.append((Path(hyp_path).parent.parent.name, Path(hyp_path).stem, logged, score))
for run_name, tag, logged, score in rows:
    print(f"{run_name:28} {tag:26} logged {logged:6.2f}  re-scored {score:6.2f}")
mismatches = [r for r in rows if r[2] != r[3]]
assert len(rows) == 36 and not mismatches, mismatches
print(f"all {len(rows)} hypothesis files reproduce their logged BLEU")

In [ ]:
# 6. Headline tables (sections 1 and 2 of the regenerated report)
with open(f"{WORK}/STUDY.md", encoding="utf-8") as fh:
    report = fh.read()
print(report[: report.index("## 3.")])
print(f"reproduced in {time.time() - T0:.0f} s")